# nbskill behavior tests

This notebook is an executable contract for the public tools documented in the other notebooks. It builds temporary notebooks, edits them, runs them, reviews them, and converts a small Python file without touching the repository notebooks.

In [ ]:
from pathlib import Path
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb

from nbskill.convert import py2nb
from nbskill.execute import exec_nb
from nbskill.foundation import cell_hash, demo_path, remove_demo_path
from nbskill.mcp import capture_call, create_mcp
from nbskill.read import read_nb, show_doc
from nbskill.review import diff_nb
from nbskill.write import update_cell, write_nb


def _find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    return start


project_root = _find_project_root()

## Build a notebook fixture

Every test below works against a temporary notebook. This mirrors the way agents should experiment: create a small fixture, prove the tool behavior, then apply the same tool to the real notebook.

In [ ]:
root = demo_path("test_nbskill_contract")
root.mkdir()
demo = root / "demo.ipynb"

_write_nb(new_nb([
    mk_cell("## Setup\nA markdown section for context.", cell_type="markdown"),
    mk_cell("value = 3\nvalue", cell_type="code"),
]), demo)

assert demo.exists()

## Reading and symbol documentation

The reader should expose useful notebook structure without raw JSON. `show_doc` should also find a symbol inside one of this repository's source notebooks and include the nearby rationale.

In [ ]:
overview = capture_call(read_nb, path=str(demo), context="overview", show_ids=True)
assert "Cell id=" in overview
assert "## Setup" in overview

doc = capture_call(show_doc, path=str(project_root / "nbs/01_read.ipynb"), symbol="read_nb", context=1, source=False)
assert "Symbol read_nb" in doc
assert "Exported definition:" in doc

## Writing and guarded updates

The write path should append parsed Markdown and code blocks. The update path should use a stable cell id and source hash, then preserve the id while replacing the source.

In [ ]:
write_nb(
    str(demo),
    "%%markdown\n## Result\nThe next cell is edited by id.\n---\n%%code\nresult = value + 4\nresult",
    export=False,
)

result_cell = next(cell for cell in _read_nb(demo).cells if "result = value + 4" in cell.source)
old_id = result_cell.id
old_hash = cell_hash(result_cell, n=None)

update_cell(
    str(demo),
    "result = value + 5\nresult",
    cell_id=old_id,
    source_hash=old_hash,
    export=False,
)

updated = next(cell for cell in _read_nb(demo).cells if cell.id == old_id)
assert "value + 5" in updated.source

## Execution and review output

Executing the fixture should store outputs in the notebook. Reviewing a repository notebook against itself should report no code-cell changes, which keeps review noise low.

In [ ]:
exec_nb(str(demo), timeout=5, show_output=False, allow_new=True)

executed = next(cell for cell in _read_nb(demo).cells if cell.id == old_id)
texts = []
for output in executed.outputs:
    data = output.get("data", {})
    if "text/plain" in data:
        text = data["text/plain"]
        texts.append("".join(text) if isinstance(text, list) else str(text))
assert any("8" in text for text in texts)

diff_text = capture_call(diff_nb, path=str(project_root / "nbs/02_write.ipynb"), ref_a=None)
assert "No code cell changes" in diff_text

## Conversion and MCP construction

The converter should create a valid nbdev notebook from Python source, and the MCP factory should be testable without starting a server.

In [ ]:
sample_py = root / "sample_tool.py"
sample_py.write_text("def double(x):\n    return x * 2\n", encoding="utf-8")
converted = root / "sample_tool.ipynb"

py2nb(str(sample_py), dest=str(converted))
converted_nb = _read_nb(converted)
assert converted.exists()
assert "#| default_exp sample_tool" in converted_nb.cells[0].source
assert any("def double" in cell.source for cell in converted_nb.cells)

mcp = create_mcp()
assert mcp is not None

remove_demo_path(root)

## MCP concurrency and wrapper failures

MCP tools should preserve notebook locking when clients issue parallel calls, and a failing wrapper should not leave the server unable to handle a later call for the same notebook.

In [ ]:
import asyncio
import threading
import time
import nbskill.mcp as _mcp_mod

parallel_root = demo_path("test_mcp_parallel_calls")
try:
    parallel_root.mkdir()
    same_path = parallel_root / "same.ipynb"
    other_path = parallel_root / "other.ipynb"
    third_path = parallel_root / "third.ipynb"
    for path in (same_path, other_path, third_path):
        _write_nb(new_nb([mk_cell("value = 1", cell_type="code")]), path)

    guard = threading.Lock()
    state = {"active_by_path": {}, "max_by_path": {}}
    old_read_nb = _mcp_mod._read_nb

    def fake_read_nb(**kwargs):
        path = kwargs["path"]
        with guard:
            state["active_by_path"][path] = state["active_by_path"].get(path, 0) + 1
            state["max_by_path"][path] = max(state["max_by_path"].get(path, 0), state["active_by_path"][path])
        time.sleep(0.1)
        print(f"read {Path(path).name}")
        with guard:
            state["active_by_path"][path] -= 1

    try:
        _mcp_mod._read_nb = fake_read_nb
        mcp = _mcp_mod.create_mcp()
        same_results = await asyncio.gather(
            mcp.call_tool("read_nb", {"path": str(same_path)}),
            mcp.call_tool("read_nb", {"path": str(same_path)}),
        )
        assert state["max_by_path"][str(same_path)] == 1
        assert all("read same.ipynb" in result.structured_content["full_output"] for result in same_results)

        other_results = await asyncio.gather(
            mcp.call_tool("read_nb", {"path": str(other_path)}),
            mcp.call_tool("read_nb", {"path": str(third_path)}),
        )
        assert {result.structured_content["full_output"] for result in other_results} == {"read other.ipynb", "read third.ipynb"}
    finally:
        _mcp_mod._read_nb = old_read_nb
finally:
    remove_demo_path(parallel_root)

In [ ]:
import nbskill.mcp as _mcp_mod

failure_root = demo_path("test_mcp_wrapper_failure")
try:
    failure_root.mkdir()
    failure_path = failure_root / "failure.ipynb"
    _write_nb(new_nb([mk_cell("value = 1", cell_type="code")]), failure_path)

    state = {"calls": 0}
    old_read_nb = _mcp_mod._read_nb

    def failing_then_recovered(**kwargs):
        state["calls"] += 1
        if state["calls"] == 1:
            raise RuntimeError("intentional wrapper failure")
        print("recovered")

    try:
        _mcp_mod._read_nb = failing_then_recovered
        mcp = _mcp_mod.create_mcp()
        try:
            await mcp.call_tool("read_nb", {"path": str(failure_path)})
        except Exception as exc:
            assert "intentional wrapper failure" in str(exc)
        else:
            raise AssertionError("read_nb failure should propagate to the MCP caller")

        recovered = await mcp.call_tool("read_nb", {"path": str(failure_path)})
        assert "recovered" in recovered.structured_content["full_output"]
        assert state["calls"] == 2
    finally:
        _mcp_mod._read_nb = old_read_nb
finally:
    remove_demo_path(failure_root)